# Custom Eval — Candidate Review

Interactive reviewer for the **published candidate set**. It prefers
`eval/custom/candidates_verified.csv` (the 150-item mechanically gated set)
and falls back to `candidates.csv` if you are reviewing a fresh bootstrap.

For each candidate you:

1. read the **source page text** for the question,
2. fix the **question / answer / evidence quote** if needed,
3. set **status** to one of `accept`, `edit`, or `reject`.

The published set is already 90 single-page + 60 ablation-verified cross-page
(ratio 0.67). Your job is quality, not volume: spot-check ~30 across papers
and both types. Until you touch an item its provenance says `machine_gate`.

Edits autosave as you navigate. When you are done, run the **Finalize** cell
to rewrite `eval/custom/test.jsonl` with `provenance.review = human` on the
rows you edited.

Run the cells top to bottom. If the interactive widgets don't render, run the
install line in the setup cell, then restart the kernel.

In [1]:
# --- Setup ---
# If the interactive widgets below don't render, run this once, then restart the kernel:
# %pip install ipywidgets pandas

import json
from pathlib import Path

import pandas as pd

# Resolve paths whether the kernel started in the repo root or in eval/.
HERE = Path.cwd()
ROOT = HERE if (HERE / "scripts").exists() else HERE.parent
CANDIDATES_VERIFIED = ROOT / "eval" / "custom" / "candidates_verified.csv"
CANDIDATES_RAW = ROOT / "eval" / "custom" / "candidates.csv"
CANDIDATES = CANDIDATES_VERIFIED if CANDIDATES_VERIFIED.exists() else CANDIDATES_RAW
PAGE_TEXTS = ROOT / "eval" / "custom" / "page_texts.jsonl"
TEST_OUT = ROOT / "eval" / "custom" / "test.jsonl"

assert CANDIDATES.exists(), (
    f"no candidates CSV at {CANDIDATES_VERIFIED} or {CANDIDATES_RAW} -- "
    "run verify_candidates.py (or bootstrap_eval_set.py) first."
)
print("project root:", ROOT)
print("candidates  :", CANDIDATES)
print("page_texts  :", PAGE_TEXTS, "(exists:", PAGE_TEXTS.exists(), ")")

project root: c:\Users\mange\OneDrive\Desktop\feeling_creative
candidates  : c:\Users\mange\OneDrive\Desktop\feeling_creative\eval\custom\candidates.csv
page_texts  : c:\Users\mange\OneDrive\Desktop\feeling_creative\eval\custom\page_texts.jsonl (exists: True )


In [ ]:
# --- Load candidates + cached page text ---
df = pd.read_csv(CANDIDATES, dtype=str).fillna("")
df["status"] = df["status"].str.strip().str.lower().replace({"": "pending"})

page_text: dict[tuple[str, int], str] = {}
if PAGE_TEXTS.exists():
    with open(PAGE_TEXTS, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            page_text[(str(row["paper_id"]), int(row["page"]))] = row["text"]


def context_for(row) -> str:
    """Full source text for the page(s) a candidate references."""
    pages = [p.strip() for p in str(row["pages"]).split(",") if p.strip()]
    parts = []
    for p in pages:
        txt = page_text.get((str(row["paper_id"]), int(p)), "(full page text not cached)")
        parts.append(f"--- page {p} ---\n{txt}")
    return "\n\n".join(parts)

print(f"Loaded {len(df)} candidates from {CANDIDATES.name}")

Loaded 437 candidates from candidates.csv


In [3]:
# --- Live summary (run anytime to see progress) ---
def summary(df: pd.DataFrame) -> None:
    accepted = df[df["status"].isin(["accept", "edit"])]
    n_single = int((accepted["question_type"] == "single_page").sum())
    n_cross = int((accepted["question_type"] == "cross_page").sum())
    ratio = (n_cross / n_single) if n_single else float("inf")
    print("Candidates by type  :", df["question_type"].value_counts().to_dict())
    print("Candidates by status:", df["status"].value_counts().to_dict())
    print("-" * 60)
    print(f"ACCEPTED: {len(accepted)}  (single={n_single}, cross={n_cross})")
    print(f"cross/single ratio  : {ratio:.2f}   (target >= 0.50)")
    print(f"distinct papers     : {accepted['paper_id'].nunique()} / {df['paper_id'].nunique()}")
    if len(accepted) >= 150 and ratio >= 0.5:
        print(">> Target reached. Run the Finalize cell.")


summary(df)

Candidates by type  : {'single_page': 289, 'cross_page': 148}
Candidates by status: {'pending': 437}
------------------------------------------------------------
ACCEPTED: 0  (single=0, cross=0)
cross/single ratio  : inf   (target >= 0.50)
distinct papers     : 0 / 15


## Interactive reviewer

Run the cell below to launch the reviewer. Controls:

- **Type / Status filters** — focus on e.g. only `cross_page`, or only `pending`.
- **Q / A / Evidence / Notes** — edit in place. `evidence_quote` should be a short
  verbatim span from the source context that supports the answer.
- **Status** — `accept` (keep as-is), `edit` (kept after you fixed it), `reject`.
- **Prev / Next** — navigate; your edits autosave to `candidates.csv` on every move.
- **Save CSV** — force a save at any time.

Tip: keep accepting good `cross_page` items — the ratio target needs them.

In [4]:
# --- Interactive reviewer (ipywidgets) ---
import ipywidgets as widgets
from IPython.display import display

STATUS_OPTIONS = ["pending", "accept", "edit", "reject"]
state = {"order": list(df.index), "pos": 0}

type_filter = widgets.Dropdown(
    options=["all", "single_page", "cross_page"], value="all", description="Type:")
status_filter = widgets.Dropdown(
    options=["all"] + STATUS_OPTIONS, value="all", description="Status:")


def apply_filter():
    order = []
    for i in df.index:
        if type_filter.value != "all" and df.at[i, "question_type"] != type_filter.value:
            continue
        if status_filter.value != "all" and df.at[i, "status"] != status_filter.value:
            continue
        order.append(i)
    state["order"] = order
    state["pos"] = 0


apply_filter()

progress = widgets.HTML()
header = widgets.HTML()
context_box = widgets.Textarea(disabled=True,
                               layout=widgets.Layout(width="100%", height="260px"))
q_box = widgets.Textarea(description="Q:", layout=widgets.Layout(width="100%", height="60px"))
a_box = widgets.Textarea(description="A:", layout=widgets.Layout(width="100%", height="90px"))
ev_box = widgets.Textarea(description="Evidence:", layout=widgets.Layout(width="100%", height="60px"))
notes_box = widgets.Text(description="Notes:", layout=widgets.Layout(width="100%"))
status_btns = widgets.ToggleButtons(options=STATUS_OPTIONS, description="Status:")
prev_btn = widgets.Button(description="< Prev")
next_btn = widgets.Button(description="Next >")
save_btn = widgets.Button(description="Save CSV", button_style="primary")
msg = widgets.HTML()


def current_index():
    if not state["order"]:
        return None
    state["pos"] = max(0, min(state["pos"], len(state["order"]) - 1))
    return state["order"][state["pos"]]


def load_row():
    i = current_index()
    if i is None:
        header.value = "<b>No rows match the current filter.</b>"
        progress.value = ""
        context_box.value = q_box.value = a_box.value = ev_box.value = notes_box.value = ""
        return
    r = df.loc[i]
    progress.value = f"<b>{state['pos'] + 1} / {len(state['order'])}</b> &nbsp; (row {i})"
    header.value = (f"<b>{r['paper_id']}</b> &middot; {r['question_type']} "
                    f"&middot; pages {r['pages']}")
    context_box.value = context_for(r)
    q_box.value = r["candidate_question"]
    a_box.value = r["candidate_answer"]
    ev_box.value = r["evidence_quote"]
    notes_box.value = r["notes"]
    status_btns.value = r["status"] if r["status"] in STATUS_OPTIONS else "pending"


def stash_row():
    i = current_index()
    if i is None:
        return
    df.at[i, "candidate_question"] = q_box.value
    df.at[i, "candidate_answer"] = a_box.value
    df.at[i, "evidence_quote"] = ev_box.value
    df.at[i, "notes"] = notes_box.value
    df.at[i, "status"] = status_btns.value


def persist():
    df.to_csv(CANDIDATES, index=False)


def save_csv(_=None):
    stash_row()
    persist()
    acc = df[df["status"].isin(["accept", "edit"])]
    ns = int((acc["question_type"] == "single_page").sum())
    nc = int((acc["question_type"] == "cross_page").sum())
    ratio = (nc / ns) if ns else float("inf")
    msg.value = (f"Saved. Accepted={len(acc)} (single={ns}, cross={nc}, "
                 f"cross/single={ratio:.2f}).")


def go(delta):
    stash_row()
    persist()  # autosave on navigation
    state["pos"] += delta
    load_row()


def on_filter_change(_):
    stash_row()
    persist()
    apply_filter()
    load_row()


prev_btn.on_click(lambda _: go(-1))
next_btn.on_click(lambda _: go(1))
save_btn.on_click(save_csv)
type_filter.observe(on_filter_change, "value")
status_filter.observe(on_filter_change, "value")

load_row()
display(widgets.VBox([
    widgets.HBox([type_filter, status_filter, progress]),
    header,
    widgets.HTML("<i>Source context (read-only):</i>"),
    context_box,
    q_box, a_box, ev_box, notes_box,
    status_btns,
    widgets.HBox([prev_btn, next_btn, save_btn]),
    msg,
]))

## Finalize

When you've accepted enough items, run the cell below. It validates each
accepted/edited row (non-empty question, answer, evidence; valid pages;
cross-page rows need >= 2 pages) and writes `eval/custom/test.jsonl`. Rows that
fail validation are reported so you can fix them in the reviewer and re-run.

In [ ]:
# --- Finalize: candidates.csv -> test.jsonl ---
import subprocess
import sys

df.to_csv(CANDIDATES, index=False)  # make sure latest edits are on disk
result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "finalize_eval_set.py"),
     "--in", str(CANDIDATES), "--out", str(TEST_OUT)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

if TEST_OUT.exists():
    from collections import Counter
    rows = [json.loads(line) for line in open(TEST_OUT, encoding="utf-8")]
    by_type = Counter(r["question_type"] for r in rows)
    n_single = by_type.get("single_page", 0)
    n_cross = by_type.get("cross_page", 0)
    ratio = (n_cross / n_single) if n_single else float("inf")
    print(f"\ntest.jsonl: {len(rows)} items | single={n_single} cross={n_cross} "
          f"cross/single={ratio:.2f}")